In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "medical_insurance.csv").is_file():
    ROOT = ROOT.parent

INPUT = ROOT / "data" / "medical_insurance.csv"
OUTPUT = ROOT / "data" / "medical_insurance_cleaned.csv"

raw = pd.read_csv(INPUT)
print(f"Dữ liệu gốc: {raw.shape[0]:,} dòng, {raw.shape[1]} cột")
display(raw.head())

Dữ liệu gốc: 2,772 dòng, 7 cột


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


KIỂM TRA DỮ LIỆU

In [2]:
columns = ["age", "sex", "bmi", "children", "smoker", "region", "charges"]
numeric = ["age", "bmi", "children", "charges"]
categories = {
    "sex": ["female", "male"],
    "smoker": ["yes", "no"],
    "region": ["northeast", "northwest", "southeast", "southwest"],
}

if raw.empty or not raw.columns.is_unique or set(raw.columns) != set(columns):
    raise ValueError("Dataset rỗng hoặc không đúng 7 cột yêu cầu.")
if not all(pd.api.types.is_numeric_dtype(raw[col]) for col in numeric):
    raise ValueError("Có cột số sai kiểu dữ liệu; cần kiểm tra lại.")

# Quy tắc miền cơ bản, không phải ngưỡng phát hiện outlier.
invalid = {col: ~np.isfinite(raw[col]) for col in numeric}
invalid["age"] |= (raw["age"] < 0) | (raw["age"] % 1 != 0)
invalid["children"] |= (raw["children"] < 0) | (raw["children"] % 1 != 0)
invalid["bmi"] |= raw["bmi"] <= 0
invalid["charges"] |= raw["charges"] < 0
for col, allowed in categories.items():
    invalid[col] = ~raw[col].isin(allowed)
invalid = pd.DataFrame(invalid)[columns] & raw.notna()

quality = pd.DataFrame({
    "Kiểu dữ liệu": raw.dtypes.astype(str),
    "Giá trị thiếu": raw.isna().sum(),
    "Giá trị không hợp lệ": invalid.sum(),
    "Số giá trị khác nhau": raw.nunique(),
})
display(quality)
display(raw[numeric].describe().T)
for col in categories:
    print(col, raw[col].value_counts(dropna=False).to_dict())

duplicate_count = int(raw.duplicated().sum())
print("Dòng trùng hoàn toàn (không tính lần đầu):", duplicate_count)
if raw.isna().any().any() or invalid.any().any():
    raise ValueError("Có dữ liệu thiếu/không hợp lệ; cần xem xét trước khi tiếp tục.")

,Kiểu dữ liệu,Giá trị thiếu,Giá trị không hợp lệ,Số giá trị khác nhau
age,int64,0,0,47
sex,str,0,0,2
bmi,float64,0,0,548
children,int64,0,0,6
smoker,str,0,0,2
region,str,0,0,4
charges,float64,0,0,1337


,count,mean,std,min,25%,50%,75%,max
age,2772.0,39.109668,14.081459,18.0000,26.000,39.00000,51.0000,64.00000
bmi,2772.0,30.701349,6.129449,15.9600,26.220,30.44750,34.7700,53.13000
children,2772.0,1.101732,1.214806,0.0000,0.000,1.00000,2.0000,5.00000
charges,2772.0,13261.369959,12151.768945,1121.8739,4687.797,9333.01435,16577.7795,63770.42801


sex {'male': 1406, 'female': 1366}
smoker {'no': 2208, 'yes': 564}
region {'southeast': 766, 'southwest': 684, 'northwest': 664, 'northeast': 658}
Dòng trùng hoàn toàn (không tính lần đầu): 1435


LOẠI DÒNG TRÙNG

In [3]:
cleaned = raw.drop_duplicates(keep="first").reset_index(drop=True).copy()
print(f"Đã loại {duplicate_count:,} dòng trùng; còn {len(cleaned):,} dòng.")

Đã loại 1,435 dòng trùng; còn 1,337 dòng.


SO SÁNH TRƯỚC VÀ SAU + LƯU KẾT QUẢ

In [4]:
summary = pd.DataFrame({
    "Trước": [len(raw), raw.shape[1], int(raw.isna().sum().sum()), duplicate_count],
    "Sau": [len(cleaned), cleaned.shape[1], int(cleaned.isna().sum().sum()), int(cleaned.duplicated().sum())],
}, index=["Số dòng", "Số cột", "Ô thiếu giá trị", "Dòng trùng hoàn toàn"])
display(summary)

assert not cleaned.duplicated().any()
assert len(cleaned) == len(raw) - duplicate_count
cleaned.to_csv(OUTPUT, index=False)
pd.testing.assert_frame_equal(cleaned, pd.read_csv(OUTPUT), check_dtype=False)
print(f"Đã lưu và đọc lại kiểm tra: {OUTPUT.relative_to(ROOT)}")

,Trước,Sau
Số dòng,2772,1337
Số cột,7,7
Ô thiếu giá trị,0,0
Dòng trùng hoàn toàn,1435,0


Đã lưu và đọc lại kiểm tra: data\medical_insurance_cleaned.csv
